# LaSOT EDA (all 70 classes)
All logic lives in `src/`; configure classes and paths in `src/config.py`.
Data: `python scripts/fetch_lasot_annotations.py` (annotations + a few frames, ~0.5 GB).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():       # find repo root whether run from src/eda or notebooks
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config as C
from src.data.lasot import load_classes, read_split
from src.eda import stats, plots
pd.set_option("display.width", 160)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("data:", C.DATA_ROOT, "| classes:", C.CLASSES)

## 1. Load and validate

In [ ]:
seqs = load_classes(C.DATA_ROOT, C.CLASSES)
print(len(seqs), "sequences")
for s in seqs:
    if s.issues:
        print(s.name, "->", s.issues)

In [ ]:
seqs_df = stats.sequence_table(seqs)
frames = stats.frame_table(seqs)
seqs_df.drop(columns="nlp").head()

## 2. Target size and aspect ratio (present frames only)

In [ ]:
plots.plot_size_aspect(frames, save="size_aspect.png")
frames.groupby("cls")[["rel_area", "aspect"]].describe().T

## 3. Absent frames: full occlusion vs out-of-view

In [ ]:
plots.plot_absence(seqs_df, save="absence.png")
seqs_df.groupby("cls")[["frac_foc", "frac_oov", "frac_absent", "n_foc_episodes", "n_oov_episodes"]].mean()

In [ ]:
print("videos with FOC:", (seqs_df.n_foc > 0).sum(), "| with OOV:", (seqs_df.n_oov > 0).sum(), "| both flags on same frame:", (seqs_df.n_both > 0).sum())

## 4. Video length and resolution

In [ ]:
plots.plot_length_resolution(seqs_df, save="length_resolution.png")
seqs_df.groupby("cls").n_frames.describe()

## 5. Attributes
Official labels (`annos/*.txt`) are used once `C.ATTRIBUTE_ORDER` is set

In [ ]:
from src.data.lasot import load_attribute_matrix
derived = stats.derived_attributes(frames, seqs_df)
official = None
if C.ATTRIBUTE_DIR.exists():
    raw = load_attribute_matrix(C.ATTRIBUTE_DIR, seqs_df.seq)
    display(stats.check_attribute_columns(raw, derived))          # which file column is FOC / OV / ...?
    if C.ATTRIBUTE_ORDER:                                         # set in config.py after verifying
        official = stats.name_attribute_columns(raw, C.ATTRIBUTE_ORDER)
else:
    print("No official attributes at", C.ATTRIBUTE_DIR, "- using derived proxies only")
attr = stats.merge_official_attributes(derived, official)
counts = stats.attribute_counts(attr)
plots.plot_attribute_counts(counts, save="attribute_counts.png")
counts

## 6. Temporal density of occlusions / out-of-view

In [ ]:
dens_foc = stats.temporal_density(seqs, "full_occ")
dens_oov = stats.temporal_density(seqs, "out_view")
episodes = stats.episodes_table(seqs)
plots.plot_temporal_density(dens_foc, dens_oov, save="temporal_density.png")
plots.plot_episodes(episodes, save="episodes.png")
episodes.groupby(["cls", "kind"]).length.describe()

## 7. Reappearance after absence
Where the target comes back relative to where it vanished (in box sizes) and how much its size changed. A tracker that searches only around the last position fails when the displacement is large; this is the gap a world model is supposed to close.

In [ ]:
re = stats.reappearance_table(seqs)
plots.plot_reappearance(re, save="reappearance.png")
print(f"{len(re)} episodes | median displacement {re.disp.median():.2f} box sizes | "
      f"> 2 box sizes: {(re.disp > 2).mean():.0%} | size change > 2x: {((re.scale > 2) | (re.scale < .5)).mean():.0%}")
re.groupby("length_bin", observed=True).agg(episodes=("disp", "size"), median_disp=("disp", "median"),
                                            pct_far=("disp", lambda d: 100 * (d > 2).mean()))

Short episodes dominate (about 30% last only 1-2 frames), so occlusion metrics need a minimum episode length or a breakdown by length; see `src/eval/occlusion.py`.

In [ ]:
print("episodes <= 2 frames: %.0f%% | <= 5: %.0f%% | >= 30 (1 s): %.0f%%" % tuple(
    100 * x for x in [(re.length <= 2).mean(), (re.length <= 5).mean(), (re.length >= 30).mean()]))

## 8. Is the test set representative of train?

In [ ]:
test = read_split(C.SPLIT_FILE)
if test is None:
    print("No split file at", C.SPLIT_FILE, "- run scripts/fetch_lasot_annotations.py")
else:
    display(stats.split_summary(seqs_df, test))
    prev = stats.split_attribute_prevalence(attr, test)
    plots.plot_split_attributes(prev, save="split_attributes.png")
    re_test = re[re.seq.isin(test)]
    print(f"test: {len(re_test)} reappearance episodes, {(re_test.length >= 30).sum()} of them >= 1 s")
    display(prev)

Per class x attribute there are only 4 test videos per cell, so zeros there are mostly noise; the table below is kept for reference.

In [ ]:
if test is not None:
    cov = stats.coverage_table(attr, test)
    print("attributes with no test video at all:", [a for a in cov.columns if cov[a].sum() == 0])
    print("class x attribute cells with 0 test videos:", int((cov == 0).sum().sum()), "of", cov.size)

## 9. Visual sanity check of annotations

In [ ]:
plots.show_samples(seqs[0], n=4)

## 10. Save tables (small CSVs can be committed)

In [ ]:
seqs_df.to_csv(C.RESULTS_DIR / "sequences.csv", index=False)
counts.to_csv(C.RESULTS_DIR / "attribute_counts.csv")
episodes.to_csv(C.RESULTS_DIR / "episodes.csv", index=False)
attr.to_csv(C.RESULTS_DIR / "attributes_per_sequence.csv")
re.to_csv(C.RESULTS_DIR / "reappearance.csv", index=False)
if test is not None:
    prev.to_csv(C.RESULTS_DIR / "split_attribute_prevalence.csv")